Install dependencies from uv and setup reloading of imports.

In [1]:
!uv sync
%load_ext autoreload
%autoreload 2

Resolved 106 packages in 6ms
Checked 103 packages in 20ms


Create the agents.

In [10]:
from google.adk.agents import Agent
from google.adk.tools import google_search, agent_tool

import config
import tools
import callbacks
import instructions

weather_agent = Agent(
    name="weather_agent",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("weather-agent-instructions"),
    before_model_callback=callbacks.log_agent_name_before_callback,
    tools=[tools.get_weather, tools.get_lat_lon]
)

search_agent = Agent(name="search_agent",
                     model=config.GEMINI_MODEL,
                     instruction=instructions.get_agent_instructions("search-agent-instructions"),
                     before_model_callback=callbacks.log_agent_name_before_callback,
                     tools=[google_search])

root_agent = Agent(
    name="root_agent",
    model=config.GEMINI_MODEL,
    mode="chat",
    instruction=instructions.get_agent_instructions("challenge-3-router-agent-instructions"),
    before_model_callback=callbacks.log_agent_name_before_callback,
    tools=[agent_tool.AgentTool(agent=search_agent)],
    sub_agents=[weather_agent]
)

Setup the agent tester.

In [14]:
import agent_tester

tester = agent_tester.AgentTester(root_agent)

In [15]:
print("================ New York ===========================")
await tester.run_prompt("What is the weather for New York City, New York")
print("================ Reston =============================")
await tester.run_prompt("What is the weather for Reston, VA")
print("================ Los Angeles ========================")
await tester.run_prompt("What is the weather for Los Angeles, CA")

================ New York ===========================
Calling agent root_agent
Calling agent weather_agent
Calling agent weather_agent
Calling agent weather_agent


📍 Weather for New York City, New York 📅 Today                                                                    

🌤️ Conditions:     Sunny then Chance Showers And Thunderstorms 🌡️ Temperature:    High: 86°F  |  Low: 77°F 💧      
Humidity:       Not available 💨 Wind:           SW at 3 to 8 mph 🌧️ Precipitation:  40% chance of showers and     
thunderstorms (60% tonight) 👁️ Visibility:     Not available 🌅 Sunrise:        Not available 🌇 Sunset:           
Not available

================ Reston =============================
Calling agent root_agent
Calling agent weather_agent
Calling agent weather_agent
Calling agent weather_agent


📍 Weather for Reston, VA 📅 Today                                                                                 

🌤️ Conditions:     Mostly Sunny then Chance Showers and Thunderstorms 🌡️ Temperature:    High: 89°F  |  Low: 71°F  
💧 Humidity:       N/A 💨 Wind:           SW 1 to 8 mph 🌧️ Precipitation:  60% chance of showers and thunderstorms 
👁️ Visibility:     N/A 🌅 Sunrise:        N/A 🌇 Sunset:         N/A

================ Los Angeles ========================
Calling agent root_agent
Calling agent weather_agent
Calling agent weather_agent
Calling agent weather_agent


📍 Weather for Los Angeles, CA                                                                                     

🌤️ Conditions:     Sunny 🌡️ Temperature:    High: 87°F  |  Low: 67°F 💨 Wind:           0 to 10 mph (SSW) 🌧️       
Precipitation:  None expected

Perform tests for searching.

In [16]:
print("================ Time ===========================")
await tester.run_prompt("What is the current time in Tokyo")
print("================ News ===========================")
await tester.run_prompt("Tell me the top news story from yesterday.")

================ Time ===========================
Calling agent root_agent
Calling agent search_agent
Calling agent root_agent


The current time in Tokyo, Japan is 8:58 PM on Friday, August 7, 2026 (Japan Standard Time, UTC+9).

================ News ===========================
Calling agent root_agent
Calling agent search_agent
Calling agent root_agent


Here are the major news headlines from yesterday:                                                                  

🌍 International Affairs                                                                                           

 • Strait of Hormuz Negotiations: Iranian and Omani negotiators reported reaching a proposed agreement to establish
   a commercial shipping route through the Strait of Hormuz, subject to final approvals.                           
 • Middle East Tensions: Explosions in southern Lebanon resulted in casualties and retaliatory strikes amidst      
   reported ceasefire violations.                                                                                  

🇺🇸 U.S. Politics & Legal Updates                                                                                   

 • Senate Contempt Vote: The Senate Homeland Security Committee voted to hold Dr. Anthony Fauci in contempt of     
   Congress after he invoked the Fifth Amendment during testimony.                                                 
 • Aviation Safety: The FAA issued an immediate directive requiring inspections on hundreds of Boeing 737 MAX      
   aircraft over potential structural frame cracks.                                                                

-------------------------------------------------------------------------------------------------------------------

Would you like more detailed coverage on any of these stories?